# 01 - Exploratory Data Analysis
BigMart Sales: what does the raw data look like, and what needs fixing?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "train_v9rqX0R.csv").exists():
    ROOT = ROOT.parent
PLOTS = ROOT / "01_eda" / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

train = pd.read_csv(ROOT / "data" / "train_v9rqX0R.csv")
test = pd.read_csv(ROOT / "data" / "test_AbJTz2l.csv")
print("train:", train.shape, "| test:", test.shape)
train.head()

train: (8523, 12) | test: (5681, 11)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


## Missing values & dtypes

In [2]:
miss = pd.concat([train.isna().sum(), test.isna().sum()], axis=1, keys=["train", "test"])
print(miss[miss.sum(axis=1) > 0])
train.describe().T

             train    test
Item_Weight   1463   976.0
Outlet_Size   2410  1606.0


,count,mean,std,min,25%,50%,75%,max
Item_Weight,7060.0,12.857645,4.643456,4.555,8.773750,12.600000,16.850000,21.350000
Item_Visibility,8523.0,0.066132,0.051598,0.000,0.026989,0.053931,0.094585,0.328391
Item_MRP,8523.0,140.992782,62.275067,31.290,93.826500,143.012800,185.643700,266.888400
Outlet_Establishment_Year,8523.0,1997.831867,8.371760,1985.000,1987.000000,1999.000000,2004.000000,2009.000000
Item_Outlet_Sales,8523.0,2181.288914,1706.499616,33.290,834.247400,1794.331000,3101.296400,13086.964800


Missing: `Item_Weight` (~17%) and `Outlet_Size` (~28%). `Item_Visibility` also has hidden
missing values encoded as 0.

In [3]:
print("Zero visibility rows:", (train.Item_Visibility == 0).sum())
for c in ["Item_Fat_Content", "Outlet_Size", "Outlet_Type", "Outlet_Location_Type"]:
    print(f"\n{c}\n", train[c].value_counts(dropna=False))

Zero visibility rows: 526

Item_Fat_Content
 Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

Outlet_Size
 Outlet_Size
Medium    2793
NaN       2410
Small     2388
High       932
Name: count, dtype: int64

Outlet_Type
 Outlet_Type
Supermarket Type1    5577
Grocery Store        1083
Supermarket Type3     935
Supermarket Type2     928
Name: count, dtype: int64

Outlet_Location_Type
 Outlet_Location_Type
Tier 3    3350
Tier 2    2785
Tier 1    2388
Name: count, dtype: int64


`Item_Fat_Content` has inconsistent labels (LF / low fat / Low Fat, reg / Regular).

## Target distribution

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(train.Item_Outlet_Sales, bins=50, color="#4C78A8")
ax[0].set_title("Item_Outlet_Sales (right-skewed)")
ax[1].hist(np.sqrt(train.Item_Outlet_Sales), bins=50, color="#59A14F")
ax[1].set_title("sqrt(Item_Outlet_Sales) - closer to normal")
plt.tight_layout(); plt.savefig(PLOTS / "target_dist.png"); plt.show()
print("skew raw:", round(train.Item_Outlet_Sales.skew(), 2),
      "| sqrt:", round(np.sqrt(train.Item_Outlet_Sales).skew(), 2))

skew raw: 1.18 | sqrt: 0.23


C:\Users\sajss\AppData\Local\Temp\ipykernel_8852\1344201011.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(PLOTS / "target_dist.png"); plt.show()


## Numeric drivers

In [5]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(train.Item_MRP, train.Item_Outlet_Sales, s=3, alpha=.4)
ax[0].set_title("MRP vs Sales (strongest driver; 4 price bands)")
ax[1].scatter(train.Item_Visibility, train.Item_Outlet_Sales, s=3, alpha=.4, color="#E15759")
ax[1].set_title("Visibility vs Sales")
ax[2].scatter(train.Item_Weight, train.Item_Outlet_Sales, s=3, alpha=.4, color="#59A14F")
ax[2].set_title("Weight vs Sales (no signal)")
plt.tight_layout(); plt.savefig(PLOTS / "numeric_vs_sales.png"); plt.show()

C:\Users\sajss\AppData\Local\Temp\ipykernel_8852\3251423871.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(PLOTS / "numeric_vs_sales.png"); plt.show()


## Categorical drivers

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), ["Outlet_Identifier", "Outlet_Type", "Outlet_Size",
                                "Outlet_Location_Type", "Item_Type", "Item_Fat_Content"]):
    train.groupby(c).Item_Outlet_Sales.mean().sort_values().plot.barh(ax=ax, color="#4C78A8")
    ax.set_title(f"Mean sales by {c}"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(PLOTS / "categorical_vs_sales.png"); plt.show()

C:\Users\sajss\AppData\Local\Temp\ipykernel_8852\3527993520.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(PLOTS / "categorical_vs_sales.png"); plt.show()


`OUT027` (Supermarket Type3) sells most; `OUT010`/`OUT019` (grocery stores) sell least.
Outlet identity is a very strong signal.

## Correlations

In [7]:
num = train.select_dtypes("number").drop(columns="Outlet_Establishment_Year")
corr = num.corr()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.savefig(PLOTS / "corr.png"); plt.show()

C:\Users\sajss\AppData\Local\Temp\ipykernel_8852\3788414860.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.colorbar(im); plt.tight_layout(); plt.savefig(PLOTS / "corr.png"); plt.show()


## Takeaways
- Fix: fat-content labels, missing weight, missing outlet size, zero visibility.
- Target is skewed -> model on `sqrt(sales)`.
- Engineer: outlet age, item category (ID prefix), price ratios, relative visibility.